In [19]:
import pandas as pd
import numpy as np
from keras.models import load_model
import joblib

features = ['moisture_before', 'moisture_after', 'duration', 'moisture_gain', 'moisture_rate']

In [20]:
# Load scaler dan model yang sudah dilatih
scaler = joblib.load('scaler.pkl')
model = load_model('best_model.keras')
mse_threshold = 0.6523

## Data Normal

In [21]:
df_normal = pd.read_excel('data/normal_data.xlsx')
df_normal.head()

,moisture_before,moisture_after,duration,moisture_gain,moisture_rate
0,39.0,80.4,7.05,41.4,5.87
1,38.0,79.6,6.75,41.6,6.16
2,37.3,80.8,6.74,43.5,6.45
3,39.7,79.9,4.82,40.2,8.34
4,37.0,80.6,7.05,43.6,6.18


In [22]:
df_normal = df_normal[features]
df_normal.head()

,moisture_before,moisture_after,duration,moisture_gain,moisture_rate
0,39.0,80.4,7.05,41.4,5.87
1,38.0,79.6,6.75,41.6,6.16
2,37.3,80.8,6.74,43.5,6.45
3,39.7,79.9,4.82,40.2,8.34
4,37.0,80.6,7.05,43.6,6.18


In [23]:
def getSeverity(fault_ratio: float):
    if fault_ratio > 2.5:
        return 'high'
    elif fault_ratio > 1.5:
        return 'medium'
    else:
        return 'low'

In [24]:
def autoencoderDetection(df: pd.DataFrame):
    # Data yang sudah discaling
    scaler_df = scaler.transform(df)

    # Rekonstruksi dari autoencoder
    recon_data = model.predict(scaler_df, verbose=0)

    # Reconstruction error
    error_data = np.abs(scaler_df - recon_data)

    # Mean error tiap sampel
    mean_error = np.mean(error_data, axis=1)

    # Fault ratio
    fault_ratio = mean_error / mse_threshold

    # Maximum error tiap sampel
    max_error = np.max(error_data, axis=1)

    # Dominant feature
    dominant_idx = np.argmax(error_data, axis=1)
    dominant_feature = [features[i] for i in dominant_idx]

    # Dominant ratio
    dominant_ratio = max_error / np.sum(error_data, axis=1)
    # Membuat dataframe hasil
    result_df = pd.DataFrame({
        'avg_mse': np.round(mean_error, 2),
        'fault_ratio': np.round(fault_ratio, 2),
        'flag': fault_ratio >= 1,
        'dominant_feature': dominant_feature,
        'dominant_ratio': np.round(dominant_ratio, 3),
        'dominant_error': np.round(max_error, 2),
        'mse_moisture_before': np.round(error_data[:, 0], 2),
        'mse_moisture_after': np.round(error_data[:, 1], 2),
        'mse_duration': np.round(error_data[:, 2], 2),
        'mse_moisture_gain': np.round(error_data[:, 3], 2),
        'mse_moisture_rate': np.round(error_data[:, 4], 2),
    })

    # Severity
    result_df['severity'] = result_df['fault_ratio'].apply(getSeverity)

    return result_df

In [25]:
print(df_normal.columns.tolist())
print(features)

['moisture_before', 'moisture_after', 'duration', 'moisture_gain', 'moisture_rate']
['moisture_before', 'moisture_after', 'duration', 'moisture_gain', 'moisture_rate']


In [26]:
normal_result_df = autoencoderDetection(df_normal)
normal_result_df.head(3)

,avg_mse,fault_ratio,flag,dominant_feature,dominant_ratio,dominant_error,mse_moisture_before,mse_moisture_after,mse_duration,mse_moisture_gain,mse_moisture_rate,severity
0,0.14,0.22,False,moisture_rate,0.534,0.38,0.11,0.08,0.03,0.11,0.38,low
1,0.13,0.20,False,moisture_rate,0.383,0.26,0.00,0.23,0.08,0.10,0.26,low
2,0.07,0.11,False,moisture_rate,0.395,0.14,0.08,0.00,0.08,0.04,0.14,low


In [27]:
normal_result_df['flag'].value_counts()

flag
False    38
True      2
Name: count, dtype: int64

In [28]:
df_normal[normal_result_df['flag'] == True]

,moisture_before,moisture_after,duration,moisture_gain,moisture_rate
26,56.8,79.1,1.99,22.3,11.21
27,58.3,79.4,2.08,21.1,10.14


In [29]:
normal_result_df[normal_result_df['flag'] == True]

,avg_mse,fault_ratio,flag,dominant_feature,dominant_ratio,dominant_error,mse_moisture_before,mse_moisture_after,mse_duration,mse_moisture_gain,mse_moisture_rate,severity
26,1.04,1.59,True,moisture_rate,0.490,2.55,0.75,0.45,0.86,0.60,2.55,medium
27,0.97,1.49,True,moisture_rate,0.448,2.18,0.79,0.65,0.70,0.54,2.18,low


In [30]:
normal_result_df['dominant_feature'].value_counts()

dominant_feature
moisture_rate      30
duration            8
moisture_before     2
Name: count, dtype: int64

## Data Fault

In [31]:
df_fault = pd.read_excel('data/fault_data.xlsx')
df_fault.head(3)

,moisture_before,moisture_after,duration,moisture_gain,moisture_rate,fault_code,fault_name,fault_feature
0,49.2,88.2,6.62,39.0,5.89,F01,Sensor Drift,moisture_before
1,48.8,89.1,6.54,40.3,6.16,F01,Sensor Drift,moisture_before
2,50.1,90.2,6.58,40.1,6.09,F01,Sensor Drift,moisture_before


In [32]:
df_fault_category = df_fault[['fault_code', 'fault_name', 'fault_feature']]
df_fault = df_fault[features]

In [33]:
fault_result_df = autoencoderDetection(df_fault)
fault_result_df.tail(1)

,avg_mse,fault_ratio,flag,dominant_feature,dominant_ratio,dominant_error,mse_moisture_before,mse_moisture_after,mse_duration,mse_moisture_gain,mse_moisture_rate,severity
39,2.26,3.47,True,moisture_after,0.508,5.75,1.87,5.75,1.21,1.15,1.33,high


In [34]:
fault_result_df['fault_code'] = df_fault_category['fault_code']
fault_result_df['fault_name'] = df_fault_category['fault_name']
fault_result_df['fault_feature'] = df_fault_category['fault_feature']

fault_result_df.head(3)

,avg_mse,fault_ratio,flag,dominant_feature,dominant_ratio,dominant_error,mse_moisture_before,mse_moisture_after,mse_duration,mse_moisture_gain,mse_moisture_rate,severity,fault_code,fault_name,fault_feature
0,0.71,1.08,True,moisture_after,0.576,2.04,0.17,2.04,0.66,0.46,0.21,low,F01,Sensor Drift,moisture_before
1,0.77,1.18,True,moisture_after,0.574,2.22,0.12,2.22,0.64,0.56,0.32,low,F01,Sensor Drift,moisture_before
2,0.87,1.34,True,moisture_after,0.572,2.49,0.15,2.49,0.74,0.63,0.35,low,F01,Sensor Drift,moisture_before


In [35]:
fault_result_df['flag'].value_counts()

flag
True    40
Name: count, dtype: int64

In [36]:
fault_result_df['dominant_feature'].value_counts()

dominant_feature
moisture_after    32
duration           4
moisture_rate      4
Name: count, dtype: int64

In [49]:
match = (fault_result_df["dominant_feature"] == fault_result_df["fault_feature"]).sum()

print(f"Jumlah data yang cocok: {match}")
print(f"Total data: {len(fault_result_df)}")
print(f"Akurasi pencocokan: {match / len(fault_result_df) * 100:.2f}%")

Jumlah data yang cocok: 12
Total data: 40
Akurasi pencocokan: 30.00%


In [ ]:
df_match = fault_result_df[fault_result_df["dominant_feature"] == fault_result_df["fault_feature"]]
df_match[["fault_name", "fault_feature", "dominant_feature", "dominant_ratio", "dominant_error"]]

,fault_name,fault_feature,dominant_feature,avg_mse,dominant_ratio,dominant_error
8,Under Irrigation,moisture_after,moisture_after,1.30,0.514,3.35
9,Under Irrigation,moisture_after,moisture_after,1.24,0.521,3.23
10,Under Irrigation,moisture_after,moisture_after,1.37,0.511,3.49
11,Under Irrigation,moisture_after,moisture_after,1.30,0.518,3.38
12,Over Irrigation,moisture_after,moisture_after,1.17,0.429,2.50
13,Over Irrigation,moisture_after,moisture_after,1.09,0.436,2.38
14,Over Irrigation,moisture_after,moisture_after,1.20,0.428,2.56
15,Over Irrigation,moisture_after,moisture_after,1.11,0.432,2.40
16,Kesalahan Rule Base,duration,duration,1.04,0.300,1.56
17,Kesalahan Rule Base,duration,duration,1.03,0.302,1.56


In [51]:
fault_result = (
    fault_result_df.assign(match=fault_result_df["dominant_feature"] == fault_result_df["fault_feature"])
      .groupby("fault_name")["match"]
      .agg(["sum", "count"])
)

fault_result["accuracy"] = fault_result["sum"] / fault_result["count"] * 100

print(fault_result)

                                 sum  count  accuracy
fault_name                                           
Fungsi Keanggotaan Tidak Sesuai    0      4       0.0
Kebocoran Pipa                     0      4       0.0
Kesalahan Rule Base                4      4     100.0
Nozzle Tersumbat                   0      4       0.0
Over Irrigation                    4      4     100.0
Pipa tersumbat                     0      4       0.0
Sensor Drift                       0      4       0.0
Sensor Freeze                      0      4       0.0
Tekanan Air Rendah                 0      4       0.0
Under Irrigation                   4      4     100.0
